# Module 318 — GoodWords Spam Filter Baseline

This notebook implements the **Spam Filter Implementation** section only: load the UCI SMS messages, make word-count features, train Multinomial Naive Bayes, and evaluate held-out predictions. Later GoodWords sections can reuse the in-memory `vectorizer` and `classifier`. No attack logic is included here. Use the `AI` Python environment with pandas, NumPy, and scikit-learn installed.

The classifier sees numeric word counts, not message meaning as a person would understand it.

## Formula the code implements

Bayes' rule is `P(C|D) = P(D|C) P(C) / P(D)`. Read `P(C|D)` aloud as **“probability of class C given document D.”** `C` is spam or ham; `P(C)` is its prior; `P(D|C)` measures how likely the message is under that class. The model selects `arg max`—**“the argument that maximizes”**—the class score.

Naive Bayes approximates `P(D|C)` as the product, from `i=1` to `n`, of `P(w_i|C)`. Say `w_i` as **“w sub i.”** This assumes word features are conditionally independent given the class. Taking logs turns the product into the sum `log P(C) + sum_i log P(w_i|C)`, avoiding multiplication of many tiny probabilities.

With vocabulary `[free, hello]`, the text `free free hello` becomes the count vector `[2, 1]`.

In [ ]:
from io import BytesIO
from pathlib import Path
import csv
import html
import random
import re
import unicodedata
from urllib.request import urlopen
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

In [ ]:
# Fixed seeds make the random train/test split reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_PATH = DATA_DIR / 'sms_spam.csv'
print('Dataset cache:', DATASET_PATH.resolve())

## Load and cache the dataset

The first run downloads the course-linked UCI archive and writes a parsed CSV into `data/`. Reruns use that local cache. The archive is parsed in memory. Size and member-name checks catch an unexpected response before it is treated as training data.

In [ ]:
if DATASET_PATH.is_file():
    df = pd.read_csv(DATASET_PATH)
else:
    dataset_url = ('https://archive.ics.uci.edu/ml/machine-learning-databases/'
                   '00228/smsspamcollection.zip')
    with urlopen(dataset_url, timeout=30) as response:
        archive_bytes = response.read()
    if len(archive_bytes) > 5_000_000:
        raise ValueError('Unexpectedly large dataset archive')
    with ZipFile(BytesIO(archive_bytes)) as archive:
        member = archive.getinfo('SMSSpamCollection')
        if member.file_size > 2_000_000:
            raise ValueError('Unexpectedly large SMS file')
        with archive.open(member) as source:
            df = pd.read_csv(source, sep='\t', header=None,
                names=['label', 'message'], quoting=csv.QUOTE_NONE,
                encoding='utf-8')
    df.to_csv(DATASET_PATH, index=False)

assert list(df.columns) == ['label', 'message']
assert df['label'].isin(['ham', 'spam']).all()
print('Rows:', len(df))
print(df['label'].value_counts())
display(df.groupby('label', group_keys=False).head(2)[['label', 'message']])

## Normalize text but preserve useful clues

We normalize encoding, case, and whitespace while retaining numbers, currency, and selected punctuation. Aggressive cleaning can erase spam evidence before the classifier can learn it. A retained character only becomes a model feature if the vectorizer's token rules accept it.

In [ ]:
def minimal_clean(text):
    """Normalize HTML entities, Unicode, and whitespace."""
    text = html.unescape(text)
    text = unicodedata.normalize('NFKC', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_text(text):
    """Lowercase and retain common spam indicators."""
    text = text.lower()
    text = re.sub(r"[^\w\s£$€¥!?.,;:'\"-]", ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df['clean_message'] = df['message'].map(minimal_clean).map(clean_text)
before = len(df)
df = df.drop_duplicates(subset=['label', 'clean_message']).copy()
df = df[df['clean_message'].str.len() > 0].copy()
print('Rows removed:', before - len(df))
print('Rows retained:', len(df))
print('Class proportions (%):')
print((100 * df['label'].value_counts(normalize=True)).round(1))

## Split before fitting the vocabulary

The training data teaches both the vocabulary and class word statistics. The held-out test data measures performance after training. Fitting a vectorizer on all rows would expose vocabulary information from the test set. `stratify=y` keeps the class proportions similar in both groups.

In [ ]:
X_text = df['clean_message']
y = df['label']
train_text, test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, random_state=SEED, stratify=y
)
print('Training rows:', len(train_text), 'Test rows:', len(test_text))
print('Train class mix:\n', y_train.value_counts(normalize=True).round(3))
print('Test class mix:\n', y_test.value_counts(normalize=True).round(3))

## Create sparse word-count features

`fit_transform` learns token-to-column mappings from training messages and encodes their counts. `transform` uses that same mapping on test messages. A sparse matrix stores only nonzero entries instead of allocating memory for every absent word in every message.

In [ ]:
vectorizer = CountVectorizer(
    max_features=3000,
    token_pattern=r'\b\w+\b|[£$€¥]+|\d+|!!+|\?\?+|\.\.+',
    lowercase=True, stop_words='english'
)
X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)
feature_names = vectorizer.get_feature_names_out()
print('Train matrix (messages, features):', X_train.shape)
print('Test matrix (messages, features):', X_test.shape)
print('Storage format:', X_train.format)
print('Example vocabulary:', feature_names[:20].tolist())

## Train and evaluate Multinomial Naive Bayes

`fit` estimates class priors and smoothed word-count likelihoods. Accuracy is the share of all predictions that are correct. For spam, precision asks what fraction of messages predicted as spam truly are spam; recall asks what fraction of all actual spam messages the model catches. Because ham is the majority class, accuracy alone can hide poor spam detection.

In [ ]:
classifier = MultinomialNB()
classifier.fit(X_train, y_train)
test_predictions = classifier.predict(X_test)
print('Learned classes:', classifier.classes_.tolist())
print('Test accuracy:', round(accuracy_score(y_test, test_predictions), 4))
print('Confusion matrix (actual rows; predicted columns; ham, spam):')
print(confusion_matrix(y_test, test_predictions, labels=['ham', 'spam']))
print(classification_report(y_test, test_predictions,
    labels=['ham', 'spam'], target_names=['ham', 'spam'], zero_division=0))

## Baseline boundary and COAE recall

HTB's displayed accuracy and per-class metrics are run-specific examples; dataset, split, tokenization, and library versions can change them. In the validated local run, the UCI archive returned 5,574 rows (4,827 ham, 747 spam), while the HTB page shows 5,572 (4,825 ham, 747 spam); after cleaning, our run retained 5,155 versus HTB's displayed 5,153. The two-row difference is recorded rather than silently removed. This notebook learns the vocabulary from training rows only and avoids loading serialized Python pickle files. It stores the fitted `vectorizer` and `classifier` in memory for later sections.

**Remember:** Multinomial Naive Bayes adds word evidence by class; the bag-of-words representation ignores context; stratify the split; evaluate minority-class precision and recall, not accuracy alone. The next section below reuses this fitted model for the GoodWords experiment.

# Section 4 — GoodWords Attack Implementation

HTB assumes **white-box access**: we can inspect the model vocabulary and its learned word probabilities. We will rank words by their relative ham evidence, append the top candidates to held-out spam text, and measure how often the unchanged classifier predicts ham. This section uses the trained `classifier` and `vectorizer` above.

The internal matrix `feature_log_prob_[class_index, feature_index]` stores `log P(w_i | class)`. Read `w_i` as **“w sub i.”** We use `classifier.classes_` to locate the ham and spam rows rather than assuming a fixed class order.

In [ ]:
class_names = classifier.classes_.tolist()
ham_index = class_names.index('ham')
spam_index = class_names.index('spam')

# Each row is a class; each column lines up with one CountVectorizer feature.
ham_log_probs = classifier.feature_log_prob_[ham_index]
spam_log_probs = classifier.feature_log_prob_[spam_index]
assert len(feature_names) == len(ham_log_probs) == len(spam_log_probs)

# exp(log(p)) recovers p. Epsilon prevents a zero denominator in the ratio.
ham_probs = np.exp(ham_log_probs)
spam_probs = np.exp(spam_log_probs)
epsilon = 1e-10
goodness = ham_probs / (spam_probs + epsilon)
goodness_table = pd.DataFrame({
    'word': feature_names,
    'goodness': goodness,
    'ham_probability': ham_probs,
    'spam_probability': spam_probs,
}).sort_values('goodness', ascending=False, ignore_index=True)
top_good_words = goodness_table.head(100).copy()
display(top_good_words.head(10))

## What the goodness score means

The course formula is `S(w) = P(w|ham) / (P(w|spam) + epsilon)`. Say it aloud as **“S of w equals probability of w given ham divided by probability of w given spam plus epsilon.”** A score above one means the estimated word probability is larger for ham than spam. This is a ranking ratio, not a probability. A very rare feature can receive a huge ratio because its spam probability is tiny, so inspect the two absolute probabilities alongside the ratio. Another stable ranking can compare their log-probability difference, but we retain HTB's ratio here to match the lesson.

`epsilon = 1e-10` is ten to the minus ten. It keeps the denominator nonzero when the estimated spam probability is zero. The score can still become very large; epsilon prevents division by zero, it does not make a rare word automatically useful or natural.

## Augment raw held-out spam messages and measure evasion

We keep the original held-out strings in `test_text`. The transformed `X_test` is a sparse count matrix and cannot be concatenated with ordinary words. For each augmented string, the same fitted vectorizer creates a one-row sparse matrix; the classifier returns probabilities in the order shown by `classifier.classes_`.

For binary classes, `P(ham|message) > P(spam|message)` is equivalent to ham probability exceeding 0.5. Evasion rate is `evaded / total spam tested × 100%`. The zero-word point measures the model's baseline false negatives, not evasion caused by added words.

In [ ]:
spam_test_text = test_text.loc[y_test == 'spam']
if spam_test_text.empty:
    raise ValueError('The held-out split contains no spam messages')

word_counts = [0, 5, 10, 15, 20, 25, 30, 35, 40]
attack_results = []
for number_of_words in word_counts:
    selected_words = top_good_words['word'].head(number_of_words).tolist()
    evaded = 0

    for original_message in spam_test_text:
        # Add text first, then vectorize with the original fitted vocabulary.
        if selected_words:
            augmented_message = original_message + ' ' + ' '.join(selected_words)
        else:
            augmented_message = original_message

        one_message_vector = vectorizer.transform([augmented_message])
        probabilities = classifier.predict_proba(one_message_vector)[0]
        if probabilities[ham_index] > probabilities[spam_index]:
            evaded += 1

    total = len(spam_test_text)
    evasion_rate = 100.0 * evaded / total
    attack_results.append({
        'num_words': number_of_words,
        'evaded': evaded,
        'total': total,
        'evasion_rate': evasion_rate,
    })

results_df = pd.DataFrame(attack_results)
display(results_df)

## Plot the measured attack curve

Each row of `results_df` is an observed percentage for one number of appended words. With a finite set of messages, the plot moves in steps; one additional message changes the rate by `100 / total` percentage points. HTB describes its curve as sigmoid-like, but a smooth sigmoid is not guaranteed. These horizontal guides are visual reference points, not universal security thresholds.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(results_df['num_words'], results_df['evasion_rate'],
        marker='o', linewidth=2, color='#00ff9c', label='Measured evasion')
ax.axhline(50, linestyle='--', color='#f2c94c', alpha=0.8, label='50% reference')
ax.axhline(90, linestyle='--', color='#56ccf2', alpha=0.8, label='90% reference')
ax.set(xlabel='Good words appended', ylabel='Evasion rate (%)',
       title='GoodWords attack effectiveness', ylim=(0, 100))
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

### Section 4 takeaways

- White-box access lets the attacker rank learned word statistics directly.
- Exponentiating `feature_log_prob_` recovers probabilities; `classes_` tells us the correct class-row indices.
- Append to raw text, then use the same vectorizer and classifier.
- Compare ham and spam probabilities; record the fraction of held-out spam predicted as ham.
- The notebook's measurements will differ from HTB's displayed 128-message results because the local baseline is freshly trained. Keep that local baseline fixed when comparing attack settings.

Only the reviewed GoodWords baseline and attack sections have been added. Later module sections remain unimplemented until their instructions are reviewed.

# Section 5 — Attack Analysis and Visualization

Section 4 measured the overall evasion rate. Here we measure the effect of each word on individual messages and visualize probability shifts. The single-word impact is `f(x) - f(x ⊕ w)`, where `f(x)` is spam probability and `x ⊕ w` is the same message with word `w` appended. A positive value means spam probability went down.

We will report the average change in **percentage points**. For example, a shift from 0.95 to 0.85 is 10 percentage points, not a 10% relative reduction.

In [ ]:
sample_size = min(50, len(spam_test_text))
sample_spam = spam_test_text.iloc[:sample_size].tolist()
candidate_words = top_good_words['word'].head(20).tolist()
if not sample_spam:
    raise ValueError('No held-out spam messages are available for impact analysis')

word_impact_records = []
for word in candidate_words:
    message_impacts = []
    for message in sample_spam:
        # f(x): spam probability before adding this one candidate.
        original_vector = vectorizer.transform([message])
        original_spam_probability = classifier.predict_proba(original_vector)[0, spam_index]

        # f(x ⊕ w): spam probability after appending only this candidate.
        augmented_message = message + ' ' + word
        augmented_vector = vectorizer.transform([augmented_message])
        augmented_spam_probability = classifier.predict_proba(augmented_vector)[0, spam_index]

        # Formula: I(w, x) = f(x) - f(x ⊕ w).
        message_impacts.append(original_spam_probability - augmented_spam_probability)

    # Average across messages, then multiply by 100 for percentage points.
    average_impact_points = 100 * float(np.mean(message_impacts))
    word_impact_records.append({
        'word': word,
        'average_impact_points': average_impact_points,
        'sample_size': sample_size,
    })

word_impact_df = pd.DataFrame(word_impact_records).sort_values(
    'average_impact_points', ascending=False, ignore_index=True)
display(word_impact_df)

The average is an empirical estimate over these held-out messages, not a property that must hold for every message. It is distinct from the Section 4 goodness ratio: goodness compares class-conditional word probabilities in the trained model, while impact measures a change to spam probability for this particular sample. The resulting average may be modest even for a high-goodness word.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(word_impact_df['word'], word_impact_df['average_impact_points'],
        color='#00ff9c', edgecolor='white', linewidth=0.5)
ax.invert_yaxis()  # Put the highest measured impact at the top.
ax.axvline(0, color='black', linewidth=0.8)
ax.set(xlabel='Average decrease in spam probability (percentage points)',
       title=f'Individual word impact on {sample_size} held-out spam messages')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()

## Track probabilities for individual messages

The next plot follows a few messages through several addition counts. Each bar shows `P(spam | message)`. In this binary classifier, a bar below 0.5 means ham has the larger probability. A message below 0.5 at zero added words is already a baseline false negative; it should not be attributed to the attack.

In [ ]:
sample_messages = spam_test_text.iloc[:8].tolist()
test_word_counts = [0, 5, 10, 20, 30]
probability_shift_records = []

for number_of_words in test_word_counts:
    selected_words = top_good_words['word'].head(number_of_words).tolist()
    for message_index, original_message in enumerate(sample_messages):
        if selected_words:
            augmented_message = original_message + ' ' + ' '.join(selected_words)
        else:
            augmented_message = original_message
        vector = vectorizer.transform([augmented_message])
        probabilities = classifier.predict_proba(vector)[0]
        probability_shift_records.append({
            'num_words': number_of_words,
            'message_index': message_index,
            'spam_probability': probabilities[spam_index],
            'ham_probability': probabilities[ham_index],
        })

probability_shift_df = pd.DataFrame(probability_shift_records)
display(probability_shift_df.head(10))

In [ ]:
x_positions = np.arange(len(sample_messages))
bar_width = 0.15
fig, ax = plt.subplots(figsize=(12, 5))

for group_index, number_of_words in enumerate(test_word_counts):
    rows = probability_shift_df[probability_shift_df['num_words'] == number_of_words]
    rows = rows.sort_values('message_index')
    bar_positions = x_positions + group_index * bar_width
    bars = ax.bar(bar_positions, rows['spam_probability'], bar_width,
                  label=f'{number_of_words} added words', alpha=0.85)
    # Mark predictions below 0.5 (ham wins); use ham probability for the check.
    for bar, ham_probability in zip(bars, rows['ham_probability']):
        if ham_probability > 0.5:
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.02, '✓', ha='center',
                    va='bottom', color='#008f5a', fontweight='bold')

ax.axhline(0.5, color='black', linestyle='--', alpha=0.7, label='Decision boundary')
ax.set_xticks(x_positions + bar_width * (len(test_word_counts) - 1) / 2)
ax.set_xticklabels([f'M{i + 1}' for i in range(len(sample_messages))])
ax.set(xlabel='Held-out spam message', ylabel='P(spam | message)',
       title='Spam probability after adding GoodWords', ylim=(0, 1.08))
ax.grid(axis='y', alpha=0.25)
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
plt.show()

### Section 5 takeaways

- Single-word impact is the original spam probability minus the augmented-message spam probability. A positive value is a decrease.
- Average the change across messages and multiply by 100 to report percentage points.
- Goodness ranking and measured impact answer different questions.
- The grouped probability plot shows that messages can cross the boundary at different word counts; identify messages already below 0.5 before augmentation.
- Log evidence adds, but probability changes are not generally additive. Measure combinations directly.

The notebook uses the existing fitted model, vectorizer, top-word table, raw held-out strings, and class indices. No additional model training is performed in this section.

# Section 7 — Black-Box Core Components

Section 6 described the black-box setting: we can submit messages and observe outputs, but cannot inspect the target model's weights or feature probabilities. This subsection prepares candidate words and tracks a query budget. It does not yet run an oracle attack.

The frequency calculation is `freq(w) = sum over ham messages m of count(w, m)`. Read this as **“frequency of w equals the sum, over ham messages m, of the number of times w occurs in m.”** It counts repeated occurrences, not just messages that contain a word.

We must use `train_text`, the raw cleaned message strings, with the aligned `y_train` labels. `X_train` in this notebook is already a sparse count matrix, so it is not the text corpus. The code checks alignment before pairing messages and labels.

In [ ]:
# Query setup: these are counters and containers only; no target query is sent here.
query_budget = 1000
queries_used = 0
query_log = []


def extract_ham_word_freq(messages, labels, sample_size=500):
    """Count token occurrences in the first `sample_size` ham messages.

    This follows the course's simple whitespace-tokenization example. The
    messages and labels must remain aligned: entry i in each refers to the
    same training example.
    """
    message_values = np.asarray(messages, dtype=object)
    label_values = np.asarray(labels)
    if len(message_values) != len(label_values):
        raise ValueError('messages and labels must have matching lengths')

    ham_messages = message_values[label_values == 'ham']
    limit = min(sample_size, len(ham_messages))
    frequencies = {}
    for message in ham_messages[:limit]:
        for word in str(message).split():
            # Keep 3-9 character tokens, as in HTB's example.
            if 2 < len(word) < 10:
                frequencies[word] = frequencies.get(word, 0) + 1
    return frequencies


def select_high_frequency_words(word_freq, max_words=100, min_freq=5):
    """Return frequent words, ordered by count descending then alphabetically."""
    sorted_by_frequency = sorted(
        word_freq.items(), key=lambda item: (-item[1], item[0])
    )
    # HTB's `count > min_freq` is strict: min_freq=5 requires at least 6.
    return [word for word, count in sorted_by_frequency if count > min_freq][:max_words]


def merge_with_curated(top_words, additional_candidates=None):
    """Add a small conversational vocabulary and remove duplicates."""
    if additional_candidates is None:
        additional_candidates = [
            'ok', 'cos', 'ill', 'thats', 'later', 'said', 'ask', 'didnt',
            'dont', 'doing', 'going', 'come', 'home', 'tomorrow', 'today',
            'sorry', 'thanks', 'yeah', 'yes', 'sure', 'see', 'tell', 'know',
            'think',
        ]
    # Set union guarantees one copy of each candidate.
    return sorted(set(top_words) | set(additional_candidates))


def build_candidate_vocabulary(
    messages, labels, sample_size=500, max_words=100, min_freq=5,
    additional_candidates=None,
):
    """Build a reproducible candidate list from ham text plus curated terms."""
    word_freq = extract_ham_word_freq(messages, labels, sample_size=sample_size)
    frequent_words = select_high_frequency_words(
        word_freq, max_words=max_words, min_freq=min_freq
    )
    merged = merge_with_curated(frequent_words, additional_candidates)

    # Keep frequency-ranked words first; alphabetical order breaks ties.
    return sorted(merged, key=lambda word: (-word_freq.get(word, 0), word))


## Read the helper pipeline

1. `extract_ham_word_freq` selects messages whose label is exactly `ham`, then counts whitespace-separated token occurrences. The first 500 ham rows are used, matching the course example; row order can bias this sample.
2. `select_high_frequency_words` sorts by count and retains up to 100 words above the strict frequency threshold. `(-count, word)` means “largest count first; alphabetical order for ties.”
3. `merge_with_curated` performs set union. In the final builder, unseen curated terms have frequency zero, so they come after observed terms. The final list may exceed 100 because the cap is applied before curated terms are added.

This candidate pool comes from available ham training text. A true query-only attacker may not have that corpus; in that case, use another permitted source. Being common in ham is only a hypothesis about a word, not evidence that the hidden classifier will reward it.

In [ ]:
# Use the raw strings and their aligned labels, not sparse X_train vectors.
ham_word_freq = extract_ham_word_freq(train_text, y_train, sample_size=500)
candidate_words_ordered = build_candidate_vocabulary(
    train_text, y_train, sample_size=500, max_words=100, min_freq=5
)
print(f'Ham messages sampled: {min(500, int((y_train == "ham").sum()))}')
print(f'Unique tokens counted: {len(ham_word_freq)}')
print(f'Candidates after curated merge: {len(candidate_words_ordered)}')
print('First 10 frequency-ranked candidates:', candidate_words_ordered[:10])

## Reserve and track the query budget

The course allocates 40% of queries to exploration, 40% to exploitation, and the remainder to combination experiments. For total budget `B`, say `floor(0.4 × B)` as **“round down 0.4 times B.”** With `B=1000`, the planned counts are 400, 400, and 200, which sum to 1000. This split is a heuristic; assigning the remainder to the last phase guarantees exact totals even when `B` is not divisible cleanly.

Initializing `queries_used` does not spend a query. Later oracle-call code must check remaining budget before each request and follow the service's accounting rules for errors, retries, or batches.

In [ ]:
def estimate_budget_allocation(total_budget):
    """Return a 40/40/remainder split whose integer parts sum exactly."""
    if total_budget < 0:
        raise ValueError('total_budget must be non-negative')
    exploration = int(0.4 * total_budget)
    exploitation = int(0.4 * total_budget)
    combination = total_budget - exploration - exploitation
    return {
        'exploration': exploration,
        'exploitation': exploitation,
        'combination': combination,
    }

allocation = estimate_budget_allocation(query_budget)
print('Query budget allocation:', allocation)
print('Allocated total:', sum(allocation.values()), '/', query_budget)

# Make shuffled working copies, preserving the source series and ranked list.
rng = np.random.default_rng(SEED)
candidate_words = candidate_words_ordered.copy()
rng.shuffle(candidate_words)
spam_test_messages = spam_test_text.tolist()
test_spam_samples = spam_test_messages[:50].copy()
rng.shuffle(test_spam_samples)
print(f'Prepared {len(candidate_words)} candidates and {len(test_spam_samples)} spam samples.')

### Section 7 COAE recall

- **Raw text versus features:** mine words from `train_text`; `X_train` contains numeric feature columns.
- `2 < len(word) < 10` keeps lengths 3–9. `count > 5` means at least six occurrences. Frequencies count repeated tokens.
- The top-100 limit is applied before curated additions, so the merged vocabulary can be larger than 100.
- A 40/40/20 split of 1,000 queries is 400 exploration, 400 exploitation, 200 combination. The last phase absorbs rounding.
- This section prepares candidate and query-management structures. It makes no model queries and does not yet identify effective words. Candidate frequency in ham is not a guarantee of target success.
- We use a seeded random generator to make the working order reproducible, while preserving the original ranked candidates and held-out text objects.

# Section 8 — Black-Box Adaptive Discovery Methods

Section 7 prepared a candidate vocabulary and query counter. This section learns from score changes, chooses words with epsilon-greedy exploration, updates their scores with an exponential moving average, then checks a bounded set of pairs and triplets.

**Simulation boundary:** HTB's sample accesses `classifier.predict_proba` directly. We put the local model behind a query-shaped wrapper that returns only a spam score and counts each submitted message. The adaptive code sees a score-only interface, but the implementation underneath still has access to the local classifier. Replace the wrapper body to connect a genuine authorized black-box service.

The EMA formula is `S_t = (1 - alpha) S_(t-1) + alpha r_t`. Read it aloud as **“S sub t equals one minus alpha times the previous score, plus alpha times the new reward.”** With an old score of 0.10, a new reward of 0.15, and `alpha=0.3`, the result is `0.7*0.10 + 0.3*0.15 = 0.115`.

In [ ]:
import math
from itertools import combinations


def query_spam_probability(message):
    """Return one score as if this local model were a score-only oracle.

    In this study notebook the local classifier simulates the oracle. A real
    client would replace the body with one authorized service request.
    """
    global queries_used
    if queries_used >= query_budget:
        raise RuntimeError(
            f'Query budget exhausted ({queries_used}/{query_budget})'
        )

    message_vector = vectorizer.transform([message])
    spam_probability = float(
        classifier.predict_proba(message_vector)[0][spam_index]
    )
    # Count one submitted message, not one probability value or class.
    queries_used += 1
    query_log.append({
        'query_number': queries_used,
        'message_characters': len(message),
        'spam_probability': spam_probability,
    })
    return spam_probability


## Epsilon-greedy word selection

Epsilon (`ε`, pronounced **“epsilon”**) is 0.2. A uniform random draw below 0.2 selects the exploration branch; otherwise the algorithm exploits the best observed word. When exploring, try an unseen word first, or the least-tested word after all have observations. The seeded random generator keeps this notebook's demonstration reproducible.

This contrasts with Section 6's UCB: epsilon-greedy flips a fixed-probability exploration switch, while UCB adds an uncertainty bonus to each candidate's score. The fixed switch is simpler, but it does not automatically reduce exploration as evidence accumulates.

In [ ]:
def initialize_adaptive_scorer(exploration_rate=0.2):
    """Create per-word score/count maps and the epsilon setting."""
    if not 0.0 <= exploration_rate <= 1.0:
        raise ValueError('exploration_rate must be between 0 and 1')
    return {
        'word_scores': {},
        'word_counts': {},
        'exploration_rate': exploration_rate,
    }


def epsilon_greedy_select(scorer, available_words, rng=None):
    """Choose an untested/least-tested word sometimes, else the best score."""
    if not available_words:
        raise ValueError('available_words must not be empty')
    if rng is None:
        rng = random.Random()

    if rng.random() < scorer['exploration_rate']:
        untested = [word for word in available_words
                    if word not in scorer['word_counts']]
        if untested:
            return rng.choice(untested)
        return min(available_words,
                   key=lambda word: scorer['word_counts'].get(word, 0))

    # Untested scores default to 0, as in HTB's sample. Stable input order
    # breaks ties deterministically in Python's max().
    return max(available_words,
               key=lambda word: scorer['word_scores'].get(word, 0.0))


def update_word_score(scorer, word, impact, alpha=0.3):
    """Update the word's exponential moving average and observation count."""
    if not 0.0 < alpha <= 1.0:
        raise ValueError('alpha must be greater than 0 and at most 1')
    if not math.isfinite(impact):
        raise ValueError('impact must be a finite number')

    if word not in scorer['word_scores']:
        scorer['word_scores'][word] = float(impact)
        scorer['word_counts'][word] = 1
    else:
        old_score = scorer['word_scores'][word]
        scorer['word_scores'][word] = (1 - alpha) * old_score + alpha * impact
        scorer['word_counts'][word] += 1


The new reward is the same finite difference as before: original spam score minus augmented spam score. A positive impact means the score decreased. For a fixed, deterministic example message, we cache its original score once and reuse that baseline so we do not pay for the same reference query on every trial. A real service may be stochastic or context-sensitive; its baseline/caching policy should match the service and experiment design.

In [ ]:
# Use one held-out spam example to demonstrate score updates. The label is used
# only to prepare this local evaluation sample; the scorer sees only outputs.
if not test_spam_samples:
    raise ValueError('No held-out spam messages are available for the demo')

adaptive_message = test_spam_samples[0]
adaptive_scorer = initialize_adaptive_scorer(exploration_rate=0.2)
selector_rng = random.Random(SEED)
original_spam_score = query_spam_probability(adaptive_message)
adaptive_records = []

for trial in range(10):
    word = epsilon_greedy_select(
        adaptive_scorer, candidate_words_ordered, rng=selector_rng
    )
    modified_message = adaptive_message + ' ' + word
    modified_spam_score = query_spam_probability(modified_message)
    reward = original_spam_score - modified_spam_score
    update_word_score(adaptive_scorer, word, reward, alpha=0.3)
    adaptive_records.append({
        'trial': trial + 1,
        'word': word,
        'spam_score': modified_spam_score,
        'reward': reward,
        'updated_average': adaptive_scorer['word_scores'][word],
        'times_tested': adaptive_scorer['word_counts'][word],
    })

display(pd.DataFrame(adaptive_records))
print(f'Queries used after adaptive demo: {queries_used}/{query_budget}')

## Test pairs and a limited number of triplets

For a pair, HTB computes `synergy = combined impact - (impact of word 1 + impact of word 2)`. Say **“synergy equals the pair's impact minus the sum of both individual impacts.”** If each word alone reduces the score by 0.10, while the pair reduces it by 0.25, synergy is 0.05. Store pair impact and synergy separately: impact tells how much the score fell; synergy tells how that compares with adding the two individual reductions.

Combinations ignore order. “n choose 2” counts pairs without duplicates and equals `n*(n-1)/2`; “n choose 3” counts triplets and equals `n*(n-1)*(n-2)/6`. For 20 candidates, exhaustive search would test 190 pairs and 1,140 triplets. The course narrows this: 20 singles, 105 pairs among the top 15, and up to 50 triplets extending five leading positive-synergy pairs with 10 words. That is at most 176 model queries including one baseline for this message. The function below checks that worst-case allowance before making calls and deduplicates triplets reached through different pairs.

In [ ]:
def discover_word_combinations(message, test_words, max_size=3):
    """Measure single-word/pair impacts and a shortlist of triplet impacts.

    Returns separate impact and pair-synergy dictionaries. This keeps a
    combined score drop distinct from its excess over two single-word drops.
    """
    if max_size not in (1, 2, 3):
        raise ValueError('max_size must be 1, 2, or 3')
    words = list(dict.fromkeys(test_words))
    if not words:
        raise ValueError('test_words must not be empty')

    single_words = words[:20]
    pair_words = words[:15]
    triplet_words = words[:10]
    pair_candidates = list(combinations(pair_words, 2)) if max_size >= 2 else []
    triplet_query_limit = (
        min(5, len(pair_candidates)) * len(triplet_words)
        if max_size >= 3 else 0
    )
    planned_queries = (
        1 + len(single_words) + len(pair_candidates) + triplet_query_limit
    )  # one baseline plus candidate messages; upper bound includes duplicate triplets
    remaining = query_budget - queries_used
    if planned_queries > remaining:
        raise RuntimeError(
            f'Need up to {planned_queries} queries, but only {remaining} remain'
        )

    baseline_score = query_spam_probability(message)
    impacts = {}
    synergies = {}
    for word in single_words:
        score = query_spam_probability(message + ' ' + word)
        impacts[(word,)] = baseline_score - score

    if max_size >= 2:
        for word1, word2 in pair_candidates:
            pair = (word1, word2)
            augmented = message + ' ' + word1 + ' ' + word2
            score = query_spam_probability(augmented)
            pair_impact = baseline_score - score
            impacts[pair] = pair_impact
            synergies[pair] = (
                pair_impact - impacts[(word1,)] - impacts[(word2,)]
            )

    if max_size >= 3:
        positive_pairs = [pair for pair in synergies if synergies[pair] > 0]
        top_pairs = sorted(
            positive_pairs, key=lambda pair: impacts[pair], reverse=True
        )[:5]
        tested_triplets = set()
        for pair in top_pairs:
            for word in triplet_words:
                if word in pair:
                    continue
                triplet = tuple(sorted(pair + (word,)))
                if triplet in tested_triplets:
                    continue
                tested_triplets.add(triplet)
                augmented = message + ' ' + ' '.join(triplet)
                score = query_spam_probability(augmented)
                impacts[triplet] = baseline_score - score

    return {'impacts': impacts, 'pair_synergies': synergies}


In [ ]:
# Keep this demonstration small. Five words limit the local simulation to at
# most 1 baseline + 5 singles + 10 pairs + 10 unique triplets = 26 queries.
combination_demo = discover_word_combinations(
    adaptive_message, candidate_words_ordered[:5], max_size=3
)
combination_rows = [
    {'words': ' + '.join(words), 'size': len(words), 'impact': impact,
     'pair_synergy': combination_demo['pair_synergies'].get(words)}
    for words, impact in combination_demo['impacts'].items()
]
combination_df = pd.DataFrame(combination_rows).sort_values(
    ['size', 'impact'], ascending=[True, False]
).reset_index(drop=True)
display(combination_df)
print(f'Total queries used: {queries_used}/{query_budget}')
assert queries_used <= query_budget


### Section 8 COAE recall

- Epsilon-greedy explores with fixed probability `ε`; otherwise it selects the best current reward estimate. `ε=0.2` means 20% explore and 80% exploit.
- `S_t=(1-α)S_(t-1)+αr_t` is an exponential moving average. `α` weights the newest reward; this is order-sensitive and not a plain arithmetic mean.
- Pair synergy is measured pair impact minus both single impacts. Positive is super-additive for the sampled message only.
- “n choose k” counts unordered combinations; use shortlists because exhaustive pair and triplet queries grow quickly.
- The local wrapper returns only a score to the search functions, but calls the local classifier internally; this remains a white-box simulation, not a true external black-box client.
- The wrapper counts a submitted message as one query and blocks calls after the budget. It caches the baseline for repeated adaptive trials on one deterministic message.
- The combination search reserves its worst-case query count before starting and uses five words in the demonstration.
- A fixed epsilon does not have UCB's stated logarithmic-regret guarantee. Keep course results, empirical observations, and theoretical guarantees distinct.

# Section 9 — Black-Box Discovery Algorithm

The section combines the vocabulary, adaptive scorer, score-only query wrapper, and combination helper into exploration, exploitation, and combination phases.

HTB motivates query choice with `I(Q) = H(W) - H(W | Q)`. Say it as **“information from query Q equals entropy of W before the query minus entropy of W after the result.”** `H` (entropy) is uncertainty about which words work. If four candidates are equally likely, uncertainty is `log2(4)=2` bits; if an observation narrows the possibilities to two equally likely candidates, one bit remains and one bit was removed. This is intuition only: the notebook does not compute entropy. A formal expected information gain averages the remaining entropy over possible outcomes of a chosen query.

Each single-word experiment needs two requests: the original message and the augmented message. Thus 400 exploration queries support 200 paired experiments. This implementation uses the same query wrapper throughout all phases, so budget accounting reflects each submitted message exactly.

## Phase design and course-to-code notes

- **Exploration:** sample among messages and words with epsilon-greedy selection. Report progress at 25%, 50%, 75%, and completion of the exploration allocation.
- **Exploitation:** rank the observed words, lower epsilon, and retest candidates from the leading group.
- **Combination:** pass a shortlist to the Section 8 pair/triplet helper. Since it measures one baseline plus single, pair, and triplet inputs, the shortlist size is chosen from the remaining phase budget first.

The Academy example uses `predict_proba(...)[0][1]` directly and estimates combination usage afterward. This notebook uses `spam_index` from the fitted class list and counts each request at the wrapper, including each baseline. It also reports that the local classifier is a simulator rather than a real black-box service.

In [ ]:
def three_phase_discovery(spam_messages, candidate_words, budget=1000):
    """Run three discovery phases while counting each submitted message.

    The score source is the local query_spam_probability wrapper. A true
    black-box integration would replace that wrapper, not this search logic.
    """
    global query_budget, queries_used, query_log
    if budget <= 0:
        raise ValueError('budget must be a positive integer')
    if not spam_messages:
        raise ValueError('spam_messages must not be empty')
    if not candidate_words:
        raise ValueError('candidate_words must not be empty')

    # This call starts a fresh, explicit experiment and resets its query log.
    query_budget = int(budget)
    queries_used = 0
    query_log = []
    allocation = estimate_budget_allocation(query_budget)
    scorer = initialize_adaptive_scorer(exploration_rate=0.2)
    rng = random.Random(SEED)
    phase_usage = {}

    # Phase 1: broad exploration. One comparison submits baseline + modified.
    phase_start = queries_used
    phase_budget = allocation['exploration']
    print(f'\n[*] Phase 1: Exploration (budget: {phase_budget} queries)')
    milestones = sorted({
        max(1, int(fraction * phase_budget))
        for fraction in (0.25, 0.50, 0.75, 1.0)
    })
    reported = set()
    while queries_used - phase_start + 2 <= phase_budget:
        message = rng.choice(spam_messages)
        word = epsilon_greedy_select(scorer, candidate_words, rng=rng)
        original_score = query_spam_probability(message)
        modified_score = query_spam_probability(message + ' ' + word)
        update_word_score(scorer, word, original_score - modified_score)

        phase_used = queries_used - phase_start
        for mark in milestones:
            if phase_used >= mark and mark not in reported:
                top3 = sorted(
                    scorer['word_scores'].items(),
                    key=lambda item: item[1], reverse=True
                )[:3]
                print(
                    f'  [P1 {phase_used}/{phase_budget}] '
                    f'tested_words={len(scorer["word_scores"])} | '
                    + 'top3=' + ', '.join(
                        f'{word}:{score:.3f}' for word, score in top3
                    )
                )
                reported.add(mark)
    phase_usage['exploration'] = queries_used - phase_start
    print(f'[+] Exploration complete: {phase_usage["exploration"]} queries, '
          f'{len(scorer["word_scores"])} words observed.')

    # Phase 2: exploit stronger candidates while retaining 10% exploration.
    phase_start = queries_used
    phase_budget = allocation['exploitation']
    scorer['exploration_rate'] = 0.1
    ranked_words = sorted(
        scorer['word_scores'].items(), key=lambda item: item[1], reverse=True
    )[:30]
    focused_words = [word for word, _ in ranked_words]
    print(f'\n[*] Phase 2: Exploitation (budget: {phase_budget} queries)')
    midpoint = max(1, phase_budget // 2)
    midpoint_reported = False
    while focused_words and queries_used - phase_start + 2 <= phase_budget:
        message_pool = spam_messages[:min(20, len(spam_messages))]
        message = rng.choice(message_pool)
        word = epsilon_greedy_select(
            scorer, focused_words[:15], rng=rng
        )
        original_score = query_spam_probability(message)
        modified_score = query_spam_probability(message + ' ' + word)
        update_word_score(scorer, word, original_score - modified_score)

        phase_used = queries_used - phase_start
        if not midpoint_reported and phase_used >= midpoint:
            top3 = sorted(
                scorer['word_scores'].items(),
                key=lambda item: item[1], reverse=True
            )[:3]
            print(f'  [P2 mid {phase_used}/{phase_budget}] top3=' +
                  ', '.join(f'{word}:{score:.3f}' for word, score in top3))
            midpoint_reported = True
    phase_usage['exploitation'] = queries_used - phase_start
    print(f'[+] Exploitation complete: {phase_usage["exploitation"]} queries.')

    # Phase 3: bounded combinations. Choose a shortlist whose worst-case
    # wrapper calls fit each share of the remaining combination budget.
    phase_start = queries_used
    phase_budget = allocation['combination']
    print(f'\n[*] Phase 3: Combination search (budget: {phase_budget} queries)')
    best_impacts = {}
    best_pair_synergies = {}
    impact_observations = {}
    synergy_observations = {}
    message_pool = spam_messages[:min(3, len(spam_messages))]

    ranked_words = sorted(
        scorer['word_scores'].items(), key=lambda item: item[1], reverse=True
    )
    combination_candidates = [word for word, _ in ranked_words]

    def worst_case_combo_queries(candidate_count):
        singles = min(candidate_count, 20)
        pair_count = min(candidate_count, 15)
        pairs = math.comb(pair_count, 2) if pair_count >= 2 else 0
        triplet_words = min(candidate_count, 10)
        triplets = min(5, pairs) * triplet_words
        return 1 + singles + pairs + triplets  # one baseline plus all candidates

    for message_index, message in enumerate(message_pool):
        remaining_phase_budget = phase_budget - (queries_used - phase_start)
        calls_left = len(message_pool) - message_index
        per_message_budget = remaining_phase_budget // calls_left
        max_candidates = next(
            (count for count in range(min(20, len(combination_candidates)), 0, -1)
             if worst_case_combo_queries(count) <= per_message_budget),
            0,
        )
        if max_candidates == 0:
            break

        measured = discover_word_combinations(
            message, combination_candidates[:max_candidates], max_size=3
        )
        for words, impact in measured['impacts'].items():
            impact_observations.setdefault(words, []).append(impact)
        for pair, synergy in measured['pair_synergies'].items():
            synergy_observations.setdefault(pair, []).append(synergy)

    # Average across messages rather than keeping only a lucky maximum.
    best_impacts = {
        words: float(np.mean(values))
        for words, values in impact_observations.items()
    }
    best_pair_synergies = {
        pair: float(np.mean(values))
        for pair, values in synergy_observations.items()
    }
    phase_usage['combination'] = queries_used - phase_start
    print(f'[+] Combination search complete: {phase_usage["combination"]} queries.')

    final_word_scores = sorted(
        scorer['word_scores'].items(), key=lambda item: item[1], reverse=True
    )
    return {
        'word_scores': final_word_scores,
        'combination_impacts': best_impacts,
        'pair_synergies': best_pair_synergies,
        'phase_queries': phase_usage,
        'allocation': allocation,
        'queries_used': queries_used,
        'query_log': query_log.copy(),
    }


## Run the full three-phase workflow

The run below uses the held-out spam strings and the ham-derived vocabulary from earlier cells. This is an offline experiment on the local course model; it sends no requests to an HTB machine or remote service. The same examples and random seed make the query order reproducible. Results vary if the dataset, vectorizer, model, or seed changes. The combined impact table averages each combination over the messages where it was tested, rather than keeping only its most favorable result.

In [ ]:
discovery_results = three_phase_discovery(
    spam_test_messages, candidate_words_ordered, budget=1000
)

print('Phase allocations:', discovery_results['allocation'])
print('Queries used by phase:', discovery_results['phase_queries'])
print(f'Total submitted messages: {discovery_results["queries_used"]}/1000')

top_final_words = pd.DataFrame(
    discovery_results['word_scores'][:10], columns=['word', 'average_reward']
)
display(top_final_words)

top_combinations = sorted(
    discovery_results['combination_impacts'].items(),
    key=lambda item: item[1], reverse=True
)[:10]
combination_summary = pd.DataFrame([
    {'words': ' + '.join(words), 'word_count': len(words),
     'average_impact': impact,
     'average_pair_synergy': discovery_results['pair_synergies'].get(words)}
    for words, impact in top_combinations
])
display(combination_summary)
assert discovery_results['queries_used'] <= 1000
assert discovery_results['queries_used'] == len(discovery_results['query_log'])


### Section 9 COAE recall

- HTB's `I(Q)=H(W)-H(W|Q)` explains why a useful observation reduces uncertainty. The code does not calculate entropy or information gain.
- A one-word reward comparison uses two queries: baseline and augmented message. Therefore 400 phase queries support 200 comparisons.
- The 40/40/20 split is a fixed allocation heuristic: for 1,000 queries, 400 exploration, 400 exploitation, and 200 combination.
- Carry one shared budget counter across the phases. Count each submitted message, including baselines and combination candidates.
- Use a per-phase query limit before a request; do not estimate usage from returned result counts after the fact. The last phase may leave budget unused to stay within its cap.
- Course code accesses `predict_proba` directly and assumes class index 1 is spam. This notebook uses the local score wrapper and the dynamically found `spam_index`; it is still a local white-box simulation underneath.
- The local run uses a held-out spam subset and averages combination results over sampled messages. Results are empirical and should not be confused with guaranteed attack success.

# Section 10 — Black-Box Attack Implementation

This section uses the discovered candidate ranking, measures evasion at several word counts, compares against the earlier white-box word ranking, and saves the results. The key empirical score remains `r_w(x)=f(x)-f(x with w appended)`. The white-box goodness ratio and black-box measured impact are different rankings, so they need not choose the same words.

The evasion rate is `R = 100 * E / N`. Read it aloud as **“R equals one hundred times the number evading divided by the number tested.”** If 14 of 20 augmented messages fall below the spam-score threshold of 0.5, the rate is `70%`. Always show `E` and `N`, and compare with the zero-word baseline to distinguish natural false negatives from attack-induced changes.

**Simulation note:** this study notebook uses the local trained Naive Bayes model behind `query_spam_probability`. It sends no requests to an external service. A real black-box adapter would replace the wrapper implementation and must follow its target's request format, rules, and rate limits.

## Prepare disjoint discovery and evaluation samples

The lesson's discovery slice `[:50]` overlaps its stated evaluation slice `[30:50]`. Here we use non-overlapping parts of the held-out spam strings. We also reserve evaluation queries before starting discovery: 20 examples at seven word counts require up to 140 score requests. The remaining budget is passed to Section 9's three-phase routine. This keeps both phases inside one total query allowance.

In [ ]:
TOTAL_QUERY_BUDGET = 1000
EVALUATION_WORD_COUNTS = [0, 5, 10, 15, 20, 25, 30]

if len(spam_test_messages) < 2:
    raise ValueError('Need at least two held-out spam messages to split discovery and evaluation')

# Use up to 50 examples for discovery, leaving a separate evaluation slice.
discovery_count = min(50, len(spam_test_messages) // 2)
discovery_messages = spam_test_messages[:discovery_count]
evaluation_messages = spam_test_messages[discovery_count:discovery_count + 20]
if not discovery_messages or not evaluation_messages:
    raise ValueError('The held-out spam set is too small for disjoint evaluation')

# Every message at every requested word count consumes one evaluation query.
EVALUATION_QUERY_RESERVE = len(evaluation_messages) * len(EVALUATION_WORD_COUNTS)
DISCOVERY_QUERY_BUDGET = TOTAL_QUERY_BUDGET - EVALUATION_QUERY_RESERVE
if DISCOVERY_QUERY_BUDGET <= 0:
    raise ValueError('Not enough total queries remain after reserving evaluation')

# The vocabulary was built from raw training messages in Section 7. Reuse it.
candidate_words = candidate_words_ordered.copy()
print(f'Discovery messages: {len(discovery_messages)}')
print(f'Disjoint evaluation messages: {len(evaluation_messages)}')
print(f'Reserved evaluation queries: {EVALUATION_QUERY_RESERVE}')
print(f'Discovery budget: {DISCOVERY_QUERY_BUDGET} / {TOTAL_QUERY_BUDGET}')
print('Allocation for discovery:', estimate_budget_allocation(DISCOVERY_QUERY_BUDGET))

## Run discovery with the reserved budget

The Section 9 function resets the shared request counter for its discovery run and returns its measured query count, phase usage, word estimates, and combination impacts. We then raise the wrapper's overall cap to the original total budget; the counter itself is not reset, so evaluation requests continue to accumulate against that same total.

In [ ]:
discovery_results = three_phase_discovery(
    discovery_messages, candidate_words, budget=DISCOVERY_QUERY_BUDGET
)

discovery_query_count = discovery_results['queries_used']
# Preserve the count and log from discovery; only restore the total cap.
query_budget = TOTAL_QUERY_BUDGET
print(f'\nDiscovery queries: {discovery_query_count}/{DISCOVERY_QUERY_BUDGET}')
print('Discovery phase counts:', discovery_results['phase_queries'])

discovered_words = discovery_results['word_scores']
print('Top 10 discovered words by measured average reward:')
display(pd.DataFrame(
    discovered_words[:10], columns=['word', 'average_reward']
))

combination_impacts = discovery_results['combination_impacts']
pair_synergies = discovery_results['pair_synergies']
top_discovered_combinations = sorted(
    combination_impacts.items(), key=lambda item: item[1], reverse=True
)[:10]
display(pd.DataFrame([
    {'words': ' + '.join(words), 'word_count': len(words),
     'average_impact': impact, 'average_pair_synergy': pair_synergies.get(words)}
    for words, impact in top_discovered_combinations
]))

## Measure evasion on messages not used during discovery

For each word count, the evaluation sends the augmented message through the score-only wrapper. The threshold follows HTB's example: spam probability **strictly below** 0.5 counts as evasion. The zero-word rate is the baseline. Because every count uses the same evaluation messages, subtracting the baseline evaded count shows the net change in this sample (it may be negative if some messages change back to spam).

The white-box control is evaluated on the same messages and word counts using `top_good_words`. This direct model call is a local comparison, so it does not consume black-box wrapper requests. It gives a more comparable sample than comparing the full white-box test set to a smaller black-box slice.

In [ ]:
blackbox_results = []
for number_of_words in EVALUATION_WORD_COUNTS:
    selected_words = [word for word, _ in discovered_words[:number_of_words]]
    evaded = 0

    for message in evaluation_messages:
        augmented = (message if not selected_words else
                     message + ' ' + ' '.join(selected_words))
        spam_probability = query_spam_probability(augmented)
        if spam_probability < 0.5:
            evaded += 1

    tested = len(evaluation_messages)
    blackbox_results.append({
        'num_words': number_of_words,
        'words_added': len(selected_words),
        'evaded': evaded,
        'tested': tested,
        'evasion_rate': 100.0 * evaded / tested,
    })

blackbox_df = pd.DataFrame(blackbox_results)
baseline_evaded = blackbox_results[0]['evaded']
blackbox_df['net_change_vs_baseline'] = (
    blackbox_df['evaded'] - baseline_evaded
)
display(blackbox_df)

# Local white-box comparison on the exact same evaluation messages.
whitebox_results = []
for number_of_words in EVALUATION_WORD_COUNTS:
    selected_words = top_good_words['word'].head(number_of_words).tolist()
    evaded = 0
    for message in evaluation_messages:
        augmented = (message if not selected_words else
                     message + ' ' + ' '.join(selected_words))
        vector = vectorizer.transform([augmented])
        spam_probability = float(
            classifier.predict_proba(vector)[0][spam_index]
        )
        if spam_probability < 0.5:
            evaded += 1
    whitebox_results.append({
        'num_words': number_of_words,
        'words_added': len(selected_words),
        'evaded': evaded,
        'tested': len(evaluation_messages),
        'evasion_rate': 100.0 * evaded / len(evaluation_messages),
    })

whitebox_df = pd.DataFrame(whitebox_results)
display(whitebox_df)

best_blackbox = max(blackbox_results, key=lambda row: row['evasion_rate'])
best_whitebox = max(whitebox_results, key=lambda row: row['evasion_rate'])
most_impactful_whitebox_word = word_impact_df.iloc[0]
test_accuracy = accuracy_score(y_test, test_predictions)
performance_share = (
    100.0 * best_blackbox['evasion_rate'] / best_whitebox['evasion_rate']
    if best_whitebox['evasion_rate'] > 0 else None
)
print('\nATTACK SUMMARY')
print(f'Local model test accuracy: {test_accuracy:.2%}')
print(f'Best white-box evasion: {best_whitebox["evasion_rate"]:.1f}% '
      f'@ {best_whitebox["num_words"]} words '
      f'({best_whitebox["evaded"]}/{best_whitebox["tested"]})')
print(f'Best black-box evasion: {best_blackbox["evasion_rate"]:.1f}% '
      f'@ {best_blackbox["num_words"]} words '
      f'({best_blackbox["evaded"]}/{best_blackbox["tested"]})')
if performance_share is not None:
    print(f'Black-box / white-box best-rate ratio: {performance_share:.1f}%')
print(f'White-box single-word leader: {most_impactful_whitebox_word["word"]} '
      f'({most_impactful_whitebox_word["average_impact_points"]:.2f} '
      'percentage points on the Section 5 sample)')
print(f'Total black-box requests: {queries_used}/{TOTAL_QUERY_BUDGET}')
assert queries_used <= TOTAL_QUERY_BUDGET
assert len(blackbox_results) == len(whitebox_results)

## Save reproducible results

The JSON file stores plain Python numbers and strings, not NumPy scalar objects. It includes the evaluation denominators, threshold, request counts, allocations, and top rankings so a future reader can interpret the rates. These files are generated run artifacts; inspect them locally rather than treating one run's specific words or percentages as universal course answers.

In [ ]:
import json

working_dir = Path.cwd()
if working_dir.name != 'AI_Evasion' and (working_dir / 'AI_Evasion').is_dir():
    working_dir = working_dir / 'AI_Evasion'
results_dir = working_dir / 'output'
results_dir.mkdir(parents=True, exist_ok=True)
results_path = results_dir / 'module318_section10_results.json'

results_payload = {
    'module': 318,
    'section': 10,
    'simulation': 'local Naive Bayes behind score-only wrapper',
    'configuration': {
        'total_query_budget': int(TOTAL_QUERY_BUDGET),
        'discovery_budget': int(DISCOVERY_QUERY_BUDGET),
        'evaluation_query_reserve': int(EVALUATION_QUERY_RESERVE),
        'evaluation_word_counts': [int(n) for n in EVALUATION_WORD_COUNTS],
        'evaluation_messages': int(len(evaluation_messages)),
        'spam_threshold': 0.5,
        'random_seed': int(SEED),
    },
    'queries': {
        'discovery_used': int(discovery_query_count),
        'evaluation_used': int(queries_used - discovery_query_count),
        'total_used': int(queries_used),
        'remaining': int(TOTAL_QUERY_BUDGET - queries_used),
        'discovery_by_phase': {
            key: int(value)
            for key, value in discovery_results['phase_queries'].items()
        },
    },
    'white_box': whitebox_results,
    'black_box': blackbox_results,
    'top_black_box_words': [
        {'word': str(word), 'average_reward': float(score)}
        for word, score in discovered_words[:20]
    ],
    'top_combinations': [
        {'words': [str(word) for word in words], 'average_impact': float(impact),
         'average_pair_synergy': (
             float(pair_synergies[words]) if words in pair_synergies else None
         )}
        for words, impact in top_discovered_combinations
    ],
    'top_white_box_word_impacts': [
        {'word': str(row['word']),
         'average_impact_points': float(row['average_impact_points']),
         'sample_size': int(row['sample_size'])}
        for _, row in word_impact_df.head(10).iterrows()
    ],
}

with results_path.open('w', encoding='utf-8') as result_file:
    json.dump(results_payload, result_file, indent=2, ensure_ascii=False)
print('Results saved to:', results_path.resolve())

### Section 10 COAE recall

- Empirical reward is the original spam score minus the augmented spam score. Goodness is a model-statistic ranking; impact is a measured score change.
- Evasion rate is `100 × evaded / tested`. Show the count and sample size, and report the zero-word baseline.
- Discovery and evaluation must not reuse the same messages if the goal is an unbiased final estimate.
- Evaluation requests belong in the same total budget. Reserve them before discovery and count each message submission through one wrapper.
- The local comparison uses the same evaluation slice for white-box and simulated black-box results. Its outcome does not establish universal 85–95% performance.
- The notebook writes JSON with configuration, threshold, denominators, phase counts, and results so the measurements remain interpretable.